<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI_Phase23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from pathlib import Path
import json, hashlib, os, re, ast, subprocess, sys
from datetime import datetime, timezone

ROOT = Path("/content")
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT = DRIVE_ROOT / "NeuroFHIR_ICHI2027"
P22_DIR = PROJECT / "phase22_outputs"
P23_DIR = PROJECT / "phase23_outputs"
PRIV = P23_DIR / "_PRIVATE_DO_NOT_SHARE"
P23_DIR.mkdir(parents=True, exist_ok=True)
PRIV.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def find_one(root, name):
    matches = list(root.rglob(name))
    return matches[0] if matches else None


In [6]:
from google.colab import drive
from pathlib import Path
import os
import json
import uuid

mountpoint = Path("/content/drive")

# Check whether Google Drive is actually mounted.
if not os.path.ismount(str(mountpoint)):
    # Preserve any folders accidentally created before mounting.
    if mountpoint.exists() and any(mountpoint.iterdir()):
        backup = Path("/content") / (
            "phase23_premount_backup_" + uuid.uuid4().hex[:8]
        )
        mountpoint.rename(backup)
        print("Prematurely created local folders moved to:", backup)

    mountpoint.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mountpoint))

# Re-establish paths AFTER mounting.
PROJECT = mountpoint / "MyDrive" / "NeuroFHIR_ICHI2027"
P22_DIR = PROJECT / "phase22_outputs"
P23_DIR = PROJECT / "phase23_outputs"
PRIV = P23_DIR / "_PRIVATE_DO_NOT_SHARE"

P23_DIR.mkdir(parents=True, exist_ok=True)
PRIV.mkdir(parents=True, exist_ok=True)

# Search across MyDrive, not just one folder.
matches = list(
    (mountpoint / "MyDrive").rglob(
        "ICHI2027_Phase22_SHAREABLE.json"
    )
)

if not matches:
    raise FileNotFoundError(
        "Phase 22 JSON is not in Google Drive. "
        "Upload ICHI2027_Phase22_SHAREABLE.json through "
        "the Google Drive website into "
        "MyDrive/NeuroFHIR_ICHI2027/phase22_outputs/ "
        "and rerun this cell."
    )

phase22 = matches[0]
p22 = json.loads(phase22.read_text())

assert p22.get("phase") == (
    "22_pinned_original_generic_TFL_producer_and_constants_archaeology"
)

print("Phase 22 loaded successfully:", phase22)

Prematurely created local folders moved to: /content/phase23_premount_backup_109b6398
Mounted at /content/drive
Phase 22 loaded successfully: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase22_outputs/ICHI2027_Phase22_SHAREABLE.json


In [7]:
REPO = ROOT / "ohdsi_tfl_phase23"
if REPO.exists():
    subprocess.run(["rm","-rf",str(REPO)], check=True)

url = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
subprocess.run(["git","clone","-q",url,str(REPO)], check=True)
subprocess.run(["git","-C",str(REPO),"checkout","-q","a4b7265adabf096e84b957de068e24a5ab853425"], check=True)

actual_commit = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"], text=True).strip()
assert actual_commit == "a4b7265adabf096e84b957de068e24a5ab853425"


In [8]:
import nbformat

candidates = [
    ("FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb", 30),
    ("FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb", 18),
    ("FHIRy_pyOMOP_TFL_Execution_v8.ipynb", 28),
    ("FHIRy_pyOMOP_TFL_MedicationReferenceHardening_v14_FIXED.ipynb", 14),
    ("FHIRy_pyOMOP_TFL_SubmissionHardening_v12_P0_1_P0_2.ipynb", 11),
]

def locate_notebook(name):
    m = list(REPO.rglob(name))
    return m[0] if m else None

extracted = []
for name, idx in candidates:
    path = locate_notebook(name)
    if path is None:
        extracted.append({"source_file": name, "cell_index": idx, "found": False})
        continue
    nb = nbformat.read(path, as_version=4)
    if idx >= len(nb.cells):
        extracted.append({"source_file": name, "cell_index": idx, "found": True, "cell_exists": False})
        continue
    src = nb.cells[idx].get("source","")
    rec = {
        "source_file": name,
        "cell_index": idx,
        "found": True,
        "cell_exists": True,
        "cell_type": nb.cells[idx].get("cell_type"),
        "source_sha256": hashlib.sha256(src.encode()).hexdigest(),
        "source": src,
    }
    extracted.append(rec)

assert any(x.get("cell_exists") for x in extracted)


/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


In [9]:
STATUS_TOKENS = {
    "PRESERVED","NORMALIZED","TRACEABILITY_LOSS","AMBIGUOUS","UNMAPPED",
    "DROPPED","LOSS","TRANSFORMATION_LOSS"
}

def string_literals(src):
    vals = []
    try:
        tree = ast.parse(src)
        for n in ast.walk(tree):
            if isinstance(n, ast.Constant) and isinstance(n.value, str):
                vals.append(n.value)
    except Exception:
        pass
    return vals

def dict_key_literals(src):
    keys = set()
    try:
        tree = ast.parse(src)
        for n in ast.walk(tree):
            if isinstance(n, ast.Dict):
                for k in n.keys:
                    if isinstance(k, ast.Constant) and isinstance(k.value, str):
                        keys.add(k.value)
    except Exception:
        pass
    return sorted(keys)

semantic = []
for rec in extracted:
    if not rec.get("cell_exists"):
        continue
    src = rec["source"]
    lits = string_literals(src)
    statuses = sorted({x for x in lits if x in STATUS_TOKENS})
    keys = dict_key_literals(src)
    warning_literals = sorted({x for x in lits if "WARN" in x.upper() or "LOSS" in x.upper()})
    semantic.append({
        "source_file": rec["source_file"],
        "cell_index": rec["cell_index"],
        "source_sha256": rec["source_sha256"],
        "status_tokens": statuses,
        "record_keys": keys,
        "warning_or_loss_literals": warning_literals[:50],
        "mentions_source_df_index": "source_df_index" in src,
        "mentions_source_resource_type": "source_resource_type" in src,
        "mentions_target_omop_table": "target_omop_table" in src,
        "mentions_target_omop_record_id": "target_omop_record_id" in src,
        "mentions_is_mapping_event": "is_mapping_event" in src,
        "mentions_warning_code": "warning_code" in src,
    })


In [10]:
required_contract = {
    "source_resource_type",
    "source_df_index",
    "target_omop_table",
    "target_omop_record_id",
    "fidelity_status",
    "warning_code",
}

for s in semantic:
    s["contract_field_count"] = sum(
        1 for k in required_contract if (
            k in s["record_keys"] or
            (k == "source_df_index" and s["mentions_source_df_index"]) or
            (k == "source_resource_type" and s["mentions_source_resource_type"]) or
            (k == "target_omop_table" and s["mentions_target_omop_table"]) or
            (k == "target_omop_record_id" and s["mentions_target_omop_record_id"]) or
            (k == "warning_code" and s["mentions_warning_code"]) or
            (k == "fidelity_status" and "fidelity_status" in s["record_keys"])
        )
    )

ranked = sorted(
    semantic,
    key=lambda x: (
        x["contract_field_count"],
        "PRESERVED" in x["status_tokens"] or "NORMALIZED" in x["status_tokens"],
        x["mentions_is_mapping_event"],
    ),
    reverse=True
)

top = ranked[0]
top


{'source_file': 'FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb',
 'cell_index': 18,
 'source_sha256': 'd448fb9a2fa378d63cb199223d7142db627b5d6def53d2507d96fab10808262c',
 'status_tokens': ['AMBIGUOUS',
  'DROPPED',
  'NORMALIZED',
  'PRESERVED',
  'TRACEABILITY_LOSS',
  'UNMAPPED'],
 'record_keys': ['audit_rows',
  'duplicate_source_id',
  'duplicate_warning_rows',
  'fidelity_status',
  'is_mapping_event',
  'mapping_rule_id',
  'source_df_index',
  'source_reference_or_path',
  'source_resource_id',
  'source_resource_type',
  'source_value_or_code',
  'target_link_method',
  'target_omop_field',
  'target_omop_record_id',
  'target_omop_table',
  'traceability_loss_rows',
  'transformation_id',
  'transformation_version',
  'variant',
  'warning_code'],
 'warning_or_loss_literals': ['TRACEABILITY_LOSS',
  'V2 Encounter audit did not reproduce 2,393 duplicate-warning rows.',
  'W_TRACEABILITY_LOSS',
  'duplicate_warning_rows',
  'traceability_loss_rows',
  'warning_code'],
 'mentio

In [11]:
# Build a semantic adapter CONTRACT ONLY.
# This does not execute the historical producer on MRI and does not invent OMOP concepts.
adapter_contract = {
    "adapter_version": "phase23_semantic_contract_v1",
    "source_domain": "FHIR_R4_MRI_numeric_observation_common_domain",
    "target_status": "RESEARCH_CANDIDATE_NOT_OFFICIAL_MICDM",
    "required_generic_audit_fields": [
        "source_resource_type",
        "source_df_index",
        "target_omop_table",
        "target_omop_record_id",
        "fidelity_status",
        "warning_code",
    ],
    "optional_generic_audit_fields": ["is_mapping_event"],
    "MRI_semantic_constraints": {
        "person_identity": "must preserve exact phase17/18 private source linkage",
        "numeric_value": "must remain exact Decimal-equivalent",
        "date_timepoint": "must remain exact",
        "image_link": "must not be represented as preserved by generic TFL unless producer semantics explicitly support it",
        "TRACE_pair_state": "must remain outside generic common-domain metrics unless producer explicitly models it",
        "unit": "not evaluable while exact source unit attestation is absent",
        "DICOM_UIDs": "not evaluable while StudyInstanceUID/SeriesInstanceUID attestation is absent",
        "OMOP_concepts": "no invented concept ids; official mapping remains blocked",
    },
    "selected_producer_candidate": {
        "source_file": top["source_file"],
        "cell_index": top["cell_index"],
        "source_sha256": top["source_sha256"],
        "selection_basis": "best static match to v15 audit-event schema only; not yet scientific human approval",
    },
}
adapter_contract["sha256"] = hashlib.sha256(
    json.dumps(adapter_contract, sort_keys=True).encode()
).hexdigest()

(PRIV / "ICHI2027_Phase23_MRI_GENERIC_C_ADAPTER_CONTRACT_PRIVATE.json").write_text(
    json.dumps(adapter_contract, indent=2)
)


1557

In [12]:
# Save exact candidate code excerpts privately for manual inspection.
review_bundle = {
    "pinned_commit": actual_commit,
    "candidates": [
        {
            "source_file": r["source_file"],
            "cell_index": r["cell_index"],
            "source_sha256": r.get("source_sha256"),
            "source": r.get("source",""),
        }
        for r in extracted if r.get("cell_exists")
    ],
    "semantic_summary": semantic,
    "selected_candidate": adapter_contract["selected_producer_candidate"],
}
(PRIV / "ICHI2027_Phase23_ORIGINAL_GENERIC_PRODUCER_CODE_REVIEW_PRIVATE.json").write_text(
    json.dumps(review_bundle, indent=2)
)


36457

In [13]:
shareable = {
    "project": "ICHI2027_neuroimaging_fhir_omop_fidelity",
    "phase": "23_original_generic_producer_semantics_and_MRI_adapter_contract",
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PINNED_ORIGINAL_PRODUCER_CANDIDATE_SEMANTICS_REVIEW_AND_MRI_ADAPTER_CONTRACT_ONLY_NO_REAL_ARM_C_EXECUTION",
    "privacy": "Aggregate/static public-code evidence only; exact old code excerpts and private MRI linkage remain in Drive.",
    "phase22_sha256": sha256_file(phase22),
    "pinned_repo": {
        "commit_verified": actual_commit == "a4b7265adabf096e84b957de068e24a5ab853425",
        "commit": actual_commit,
    },
    "producer_candidates_reviewed": len(semantic),
    "producer_semantics": [
        {
            "source_file": s["source_file"],
            "cell_index": s["cell_index"],
            "source_sha256": s["source_sha256"],
            "status_tokens": s["status_tokens"],
            "contract_field_count": s["contract_field_count"],
            "mentions_is_mapping_event": s["mentions_is_mapping_event"],
        }
        for s in ranked
    ],
    "selected_candidate_for_private_manual_review": adapter_contract["selected_producer_candidate"],
    "adapter_contract_sha256": adapter_contract["sha256"],
    "adapter_contract_constraints": adapter_contract["MRI_semantic_constraints"],
    "gates": {
        "phase22_upstream_verified": True,
        "pinned_producer_candidate_cells_retrieved": True,
        "generic_audit_schema_fields_statically_reconciled": True,
        "one_candidate_selected_for_private_manual_review": True,
        "candidate_semantics_human_approved_as_exact_generic_C_producer": False,
        "MRI_adapter_contract_frozen_without_OMOP_code_invention": True,
        "real_MRI_generic_C_event_generator_executed": False,
        "official_MICDM_load_completed": False,
        "exact_ADNI_units_attested": False,
        "DICOM_UID_semantics_attested": False,
        "Athena_mappings_approved": False,
        "true_four_arm_comparison_completed": False,
    },
    "next_stage_gate": "MANUALLY_REVIEW_SELECTED_ORIGINAL_PRODUCER_CODE; IF SEMANTICALLY_ACCEPTABLE, EXECUTE ONLY ITS GENERIC COMMON-DOMAIN LOGIC ON REAL MRI ADAPTER WITHOUT CLAIMING IMAGE/TRACE COVERAGE"
}

out = P23_DIR / "ICHI2027_Phase23_SHAREABLE.json"
out.write_text(json.dumps(shareable, indent=2))
print(out)
print(json.dumps(shareable["selected_candidate_for_private_manual_review"], indent=2))


/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase23_outputs/ICHI2027_Phase23_SHAREABLE.json
{
  "source_file": "FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb",
  "cell_index": 18,
  "source_sha256": "d448fb9a2fa378d63cb199223d7142db627b5d6def53d2507d96fab10808262c",
  "selection_basis": "best static match to v15 audit-event schema only; not yet scientific human approval"
}
